In [1]:
from PIL import Image
import pytesseract

# Если нужно явно указать путь к Tesseract (Windows)
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

image_path = "light_scan_0.jpg"
image = Image.open(image_path)

# Получаем TSV output
tsv_data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

# tsv_data — это словарь с ключами: level, page_num, block_num, par_num, line_num, word_num, left, top, width, height, conf, text
print(tsv_data.keys())

# Пример доступа к тексту и bbox
n = len(tsv_data["text"])
for i in range(n):
    line_text = tsv_data["text"][i].strip()
    if line_text:
        x = tsv_data["left"][i]
        y = tsv_data["top"][i]
        w = tsv_data["width"][i]
        h = tsv_data["height"][i]
        print({
            "text": line_text,
            "bbox": [x, y, x + w, y + h],
            "conf": tsv_data["conf"][i],
            "level": tsv_data["level"][i]  # 4 = line, 5 = word
        })

dict_keys(['level', 'page_num', 'block_num', 'par_num', 'line_num', 'word_num', 'left', 'top', 'width', 'height', 'conf', 'text'])
{'text': 'For', 'bbox': [141, 110, 179, 130], 'conf': 96, 'level': 5}
{'text': 'other', 'bbox': [188, 110, 253, 130], 'conf': 96, 'level': 5}
{'text': 'uses,', 'bbox': [263, 114, 324, 132], 'conf': 93, 'level': 5}
{'text': 'seeLinear', 'bbox': [336, 110, 456, 130], 'conf': 92, 'level': 5}
{'text': 'regre-', 'bbox': [466, 114, 539, 135], 'conf': 91, 'level': 5}
{'text': 'ssion', 'bbox': [85, 138, 146, 158], 'conf': 91, 'level': 5}
{'text': '(disambiguation).', 'bbox': [157, 138, 374, 162], 'conf': 96, 'level': 5}
{'text': 'Instatistics,', 'bbox': [141, 183, 278, 216], 'conf': 90, 'level': 5}
{'text': 'linear', 'bbox': [285, 183, 352, 216], 'conf': 90, 'level': 5}
{'text': 'regressionis', 'bbox': [362, 187, 511, 211], 'conf': 92, 'level': 5}
{'text': 'amodelthat', 'bbox': [85, 215, 227, 235], 'conf': 88, 'level': 5}
{'text': 'estimates', 'bbox': [236, 215, 35

In [2]:
from PIL import Image
import pytesseract


def extract_ocr_lines(image_path: str, zoom: float = 1.0, page_number: int = 1, debug: bool = True):
    image = Image.open(image_path)
    img_width, img_height = image.size

    tsv_data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    lines = {}
    n = len(tsv_data["text"])

    for i in range(n):
        if tsv_data["level"][i] == 5:  # слова

            word = tsv_data["text"][i].strip()
            conf = int(tsv_data["conf"][i])

            if not word:
                continue

            block = tsv_data["block_num"][i]
            par = tsv_data["par_num"][i]
            line = tsv_data["line_num"][i]
            key = (block, par, line)

            left = tsv_data["left"][i]
            top = tsv_data["top"][i]
            w = tsv_data["width"][i]
            h = tsv_data["height"][i]

            # --- координаты ---
            # image space → bottom-left
            x0_img = left
            y0_img = img_height - (top + h)

            # → PDF space
            x0 = x0_img / zoom
            y0 = y0_img / zoom
            w = w / zoom
            h = h / zoom

            if key not in lines:
                lines[key] = {
                    "words": [],
                    "conf_sum": 0,
                    "conf_count": 0,
                    "bbox": [x0, y0, w, h]
                }

            # --- добавляем слово ---
            lines[key]["words"].append(word)

            if conf > 0:
                lines[key]["conf_sum"] += conf
                lines[key]["conf_count"] += 1

            # --- расширяем bbox ---
            bx, by, bw, bh = lines[key]["bbox"]

            new_x0 = min(bx, x0)
            new_y0 = min(by, y0)

            new_x1 = max(bx + bw, x0 + w)
            new_y1 = max(by + bh, y0 + h)

            lines[key]["bbox"] = [
                new_x0,
                new_y0,
                new_x1 - new_x0,
                new_y1 - new_y0
            ]

    # --- финальная структура ---
    ocr_lines = []
    line_id = 0

    for key, line in lines.items():

        text = " ".join(line["words"])

        if not text.strip():
            continue

        conf = (
            line["conf_sum"] / line["conf_count"]
            if line["conf_count"] > 0 else 0
        )

        result = {
            "id": f"ocr_line_{line_id}",
            "type": "text_line",
            "text": text,
            "bbox": {
                "x": round(line["bbox"][0], 2),
                "y": round(line["bbox"][1], 2),
                "width": round(line["bbox"][2], 2),
                "height": round(line["bbox"][3], 2)
            },
            "confidence": round(conf, 2),
            "page_number": page_number
        }

        ocr_lines.append(result)
        line_id += 1

        # --- debug print ---
        if debug:
            print(result)

    return ocr_lines

In [3]:
ocr_lines = extract_ocr_lines(
    image_path="light_scan_0.jpg",
    zoom=2.777,
    page_number=1,
    debug=True
)

{'id': 'ocr_line_0', 'type': 'text_line', 'text': 'For other uses, seeLinear regre-', 'bbox': {'x': 50.77, 'y': 793.66, 'width': 143.32, 'height': 9.0}, 'confidence': 93.6, 'page_number': 1}
{'id': 'ocr_line_1', 'type': 'text_line', 'text': 'ssion (disambiguation).', 'bbox': {'x': 30.61, 'y': 783.94, 'width': 104.07, 'height': 8.64}, 'confidence': 93.5, 'page_number': 1}
{'id': 'ocr_line_2', 'type': 'text_line', 'text': 'Instatistics, linear regressionis', 'bbox': {'x': 50.77, 'y': 764.49, 'width': 133.24, 'height': 11.88}, 'confidence': 90.67, 'page_number': 1}
{'id': 'ocr_line_3', 'type': 'text_line', 'text': 'amodelthat estimates the relations-', 'bbox': {'x': 30.61, 'y': 757.65, 'width': 160.97, 'height': 7.2}, 'confidence': 92.0, 'page_number': 1}
{'id': 'ocr_line_4', 'type': 'text_line', 'text': 'hip between ascalarresponse (depe-', 'bbox': {'x': 30.61, 'y': 746.13, 'width': 163.49, 'height': 9.0}, 'confidence': 93.0, 'page_number': 1}
{'id': 'ocr_line_5', 'type': 'text_line', 't

In [4]:
for line in ocr_lines:
    print(line['text'])

For other uses, seeLinear regre-
ssion (disambiguation).
Instatistics, linear regressionis
amodelthat estimates the relations-
hip between ascalarresponse (depe-
ndent variable) and one or more
explanatory variables (regressororin-
dependent variable). A model with
exactly one explanatory variable is
asimple linear regression; a model
with two or more explanatory variab-
les is amultiple linear regression.[1]-
This term is distinct frommultivariate
linear regression, which predicts
multiplecorrelateddependent
variables rather than a single depen-
dent variable.[2]
In linear regression, the relation-
ships are modeled usinglinear predi-
ctor functionswhose unknown mode-
lparametersareestimatedfrom theda-
ta. Most commonly, theconditional
meanof the response given the val-
ues of the explanatory variables (or
predictors) is assumed to be anaffine
functionof those values; less commo-
nly, the conditionalmedianor some
otherquantileis used. Like all forms
ofregression analysis, linear regre

In [5]:
import json

def load_gt(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [22]:
def extract_gt_lines(gt_json, page_number):
    lines = []

    for page in gt_json["pages"]:
        if int(page["page_number"]) != int(page_number):
            continue

        for container in page["containers"]:
            for el in container["elements"]:
                if el["type"] != "text_line":
                    continue

                # 🔥 КРИТИЧЕСКАЯ ПРОВЕРКА
                if int(el["bbox"]["page"]) != int(page_number):
                    continue

                lines.append({
                    "id": el["id"],
                    "text": el["content"],
                    "bbox": el["bbox"],
                    "container_id": container["id"]  # полезно для structure позже
                })

    return lines

In [23]:
gt_json = load_gt("document_20260325_150507_gt.json")

gt_lines = extract_gt_lines(gt_json, page_number=1)

In [24]:
def compute_iou(b1, b2):
    x1_min, y1_min = b1["x"], b1["y"]
    x1_max, y1_max = x1_min + b1["width"], y1_min + b1["height"]

    x2_min, y2_min = b2["x"], b2["y"]
    x2_max, y2_max = x2_min + b2["width"], y2_min + b2["height"]

    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    if inter_xmax <= inter_xmin or inter_ymax <= inter_ymin:
        return 0.0

    inter_area = (inter_xmax - inter_xmin) * (inter_ymax - inter_ymin)

    area1 = b1["width"] * b1["height"]
    area2 = b2["width"] * b2["height"]

    union = area1 + area2 - inter_area

    return inter_area / union

In [25]:
def match_lines(gt_lines, ocr_lines, iou_threshold=0.3):
    matches = []

    for gt in gt_lines:
        best_iou = 0
        best_pred = None

        for pred in ocr_lines:
            iou = compute_iou(gt["bbox"], pred["bbox"])
            if iou > best_iou:
                best_iou = iou
                best_pred = pred

        if best_iou >= iou_threshold:
            matches.append((gt, best_pred, best_iou))
        else:
            matches.append((gt, None, 0))

    return matches

In [26]:
def assign_columns(ocr_lines, num_columns):
    xs = [l["bbox"]["x"] + l["bbox"]["width"] / 2 for l in ocr_lines]
    min_x, max_x = min(xs), max(xs)
    width = max_x - min_x

    for l in ocr_lines:
        x_center = l["bbox"]["x"] + l["bbox"]["width"] / 2
        rel = (x_center - min_x) / width

        col = int(rel * num_columns)
        if col == num_columns:
            col -= 1

        l["column"] = col

    return ocr_lines

In [27]:
def sort_ocr_lines_reading_order(ocr_lines):
    return sorted(
        ocr_lines,
        key=lambda l: (
            l["column"],
            -l["bbox"]["y"],
            l["bbox"]["x"]
        )
    )

In [28]:
def sort_gt_lines_reading_order(gt_lines):
    return sorted(
        gt_lines,
        key=lambda l: (
            l["container_id"],
            -l["bbox"]["y"]
        )
    )

In [29]:
def build_order_sequences(gt_lines, ocr_lines, num_columns):
    # 1. сортируем GT (истина)
    gt_sorted = sort_gt_lines_reading_order(gt_lines)
    gt_order = [l["id"] for l in gt_sorted]

    # 2. назначаем колонки OCR
    ocr_lines = assign_columns(ocr_lines, num_columns)

    # 3. сортируем OCR
    ocr_sorted = sort_ocr_lines_reading_order(ocr_lines)

    # 4. matching
    matches = match_lines(gt_sorted, ocr_sorted)

    # 5. формируем pred_order
    pred_order = []
    for gt, pred, _ in matches:
        if pred is not None:
            pred_order.append(gt["id"])

    return gt_order, pred_order

In [30]:
# def build_order_sequences(matches):
#     """
#     Возвращает:
#     gt_order_ids — правильный порядок
#     pred_order_ids — порядок OCR в координатах GT
#     """

#     # GT порядок (как есть)
#     gt_order_ids = [gt["id"] for gt, _, _ in matches]

#     # OCR порядок → сортируем по геометрии (reading order)
#     pred_items = []

#     for gt, pred, _ in matches:
#         if pred is None:
#             continue

#         pred_items.append({
#             "gt_id": gt["id"],
#             "y": pred["bbox"]["y"],
#             "x": pred["bbox"]["x"]
#         })

#     # 🔥 ключ: сортировка как чтение
#     pred_items = sorted(pred_items, key=lambda x: (-x["y"], x["x"]))

#     pred_order_ids = [item["gt_id"] for item in pred_items]

#     return gt_order_ids, pred_order_ids

In [31]:
def kendall_tau(gt_order, pred_order):
    """
    gt_order: [id1, id2, id3]
    pred_order: [id2, id1, id3]
    """

    # индекс GT
    gt_index = {id_: i for i, id_ in enumerate(gt_order)}

    # переводим pred_order в индексы GT
    pred_ranks = [gt_index[id_] for id_ in pred_order if id_ in gt_index]

    n = len(pred_ranks)
    concordant = 0
    discordant = 0

    for i in range(n):
        for j in range(i + 1, n):
            if pred_ranks[i] < pred_ranks[j]:
                concordant += 1
            else:
                discordant += 1

    total = concordant + discordant

    if total == 0:
        return 0

    return (concordant - discordant) / total

In [32]:
def pairwise_accuracy(gt_order, pred_order):
    gt_index = {id_: i for i, id_ in enumerate(gt_order)}
    pred_ranks = [gt_index[id_] for id_ in pred_order if id_ in gt_index]

    n = len(pred_ranks)
    correct = 0
    total = 0

    for i in range(n):
        for j in range(i + 1, n):
            total += 1
            if pred_ranks[i] < pred_ranks[j]:
                correct += 1

    return correct / total if total > 0 else 0

In [33]:
def evaluate_ordering(gt_lines, ocr_lines, num_columns):
    gt_order, pred_order = build_order_sequences(
        gt_lines,
        ocr_lines,
        num_columns
    )

    tau = kendall_tau(gt_order, pred_order)
    pair_acc = pairwise_accuracy(gt_order, pred_order)

    return {
        "kendall_tau": tau,
        "pairwise_accuracy": pair_acc,
        "num_matched": len(pred_order),
        "num_gt": len(gt_order)
    }

In [34]:
# def evaluate_ordering(gt_lines, ocr_lines):
#     # 1. Matching
#     matches = match_lines(gt_lines, ocr_lines)

#     # 2. Порядки
#     gt_order, pred_order = build_order_sequences(matches)

#     # 3. Метрики
#     tau = kendall_tau(gt_order, pred_order)
#     pair_acc = pairwise_accuracy(gt_order, pred_order)

#     return {
#         "kendall_tau": tau,
#         "pairwise_accuracy": pair_acc,
#         "num_matched": len(pred_order),
#         "num_gt": len(gt_order)
#     }

In [36]:
results = evaluate_ordering(
    gt_lines,
    ocr_lines,
    num_columns=3
)
print("\n=== ORDERING METRICS ===")
print("Kendall Tau:", round(results["kendall_tau"], 3))
print("Pairwise Acc:", round(results["pairwise_accuracy"], 3))
print("Matched:", results["num_matched"], "/", results["num_gt"])


=== ORDERING METRICS ===
Kendall Tau: 1.0
Pairwise Acc: 1.0
Matched: 207 / 207


In [37]:
def debug_order(gt_lines, ocr_lines, matches):
    print("\n===== DEBUG ORDER =====\n")

    # GT порядок
    print("---- GT ORDER ----")
    for i, gt in enumerate(gt_lines[:20]):
        print(f"{i:03d} | y={gt['bbox']['y']:.1f} | {gt['text'][:60]}")

    # OCR порядок (как есть)
    print("\n---- OCR RAW ORDER ----")
    for i, ocr in enumerate(ocr_lines[:20]):
        print(f"{i:03d} | y={ocr['bbox']['y']:.1f} | {ocr['text'][:60]}")

    # OCR после сортировки (твоя логика)
    sorted_ocr = sorted(ocr_lines, key=lambda l: (-l["bbox"]["y"], l["bbox"]["x"]))

    print("\n---- OCR SORTED ORDER ----")
    for i, ocr in enumerate(sorted_ocr[:20]):
        print(f"{i:03d} | y={ocr['bbox']['y']:.1f} | {ocr['text'][:60]}")

    # Сравнение matched
    print("\n---- MATCHED PAIRS ----")
    for i, (gt, pred, iou) in enumerate(matches[:20]):
        print(f"\n{i:03d}")
        print("GT :", gt["text"][:60])
        print("OCR:", pred["text"][:60] if pred else "NONE")
        print("IoU:", round(iou, 2))

In [38]:
matches = match_lines(gt_lines, ocr_lines)

In [39]:
debug_order(gt_lines, ocr_lines, matches)


===== DEBUG ORDER =====

---- GT ORDER ----
000 | y=792.1 | For other uses, seeLinear regre-
001 | y=782.2 | ssion (disambiguation).
002 | y=764.3 | Instatistics,linear regressionis
003 | y=754.4 | amodelthat estimates the relations-
004 | y=744.5 | hip between ascalarresponse (depe-
005 | y=734.6 | ndent variable) and one or more
006 | y=724.7 | explanatory variables (regressororin-
007 | y=714.8 | dependent variable). A model with
008 | y=704.9 | exactly one explanatory variable is
009 | y=695.0 | asimple linear regression; a model
010 | y=685.1 | with two or more explanatory variab-
011 | y=675.2 | les is amultiple linear regression.[1]-
012 | y=665.3 | This term is distinct frommultivariate
013 | y=655.4 | linear regression, which predicts
014 | y=645.5 | multiplecorrelateddependent
015 | y=635.6 | variables rather than a single depen-
016 | y=625.7 | dent variable.[2]
017 | y=607.8 | In linear regression, the relation-
018 | y=597.9 | ships are modeled usinglinear predi-
019 | y=

In [40]:
def debug_columns(lines):
    for l in lines[:20]:
        print(
            f"col={l.get('column')} y={round(l['bbox']['y'],1)} | {l['text'][:40]}"
        )